# Notebook 6: Does the preprocessing ablation hold for ConvNeXt-Tiny and ViT-S/16?

The full 18-pipeline ablation was run on ResNet-50. This notebook checks whether its main conclusions also hold for the two strongest architectures, using exactly the same data, folds, training settings and code as Notebook 2 (scenario E1, all images).

| Pipeline | Change from the reference K1 | Question |
|---|---|---|
| K3 | letterbox 384 px | Is the larger input, chosen for recipe F, also best for these models? |
| K6 | grayscale | Does discarding colour also hurt them? |
| K7 | HSV | Does reordering colour also hurt them? |

K1 (the reference) is **not retrained**: its fold scores are read from the Notebook 2 output, so 2 models x 3 pipelines x 5 folds = 30 trainings are needed (about 4 h on a T4).

## Inputs (Add Input -> Your Work)
| Input | Used for |
|---|---|
| Stage 0 output | images with background, labels, folds |
| Notebook 2 output, final version | K1 fold scores of ResNet-50, ConvNeXt-Tiny and ViT-S/16 (read only, nothing is copied) |
| *(only if a commit stops early)* this notebook's own previous version | finished runs are carried forward and skipped |

Accelerator: **GPU T4**. Commit with Save Version -> Save & Run All. If it prints NOT FINISHED, add this notebook's latest version as an input and commit again.

## Output
`/kaggle/working/ablation_check/` with one folder per run (same files as Notebook 2) and `ablation_check_table.csv`, the table for the paper.

## 1. Setup

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("timm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])

import os, json, time, math, random, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F_
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import v2 as T
import timm
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, confusion_matrix, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__, "| timm", timm.__version__)
if DEVICE == "cpu":
    warnings.warn("No GPU found. Turn on Accelerator: GPU (T4 or P100) in the notebook settings.")

device: cuda | torch 2.11.0+cu130 | timm 1.0.30


## 2. Configuration (nothing to edit)

In [2]:
SCENARIO = "all"                                  # E1: all images

CHECK_MODELS = ["convnext_tiny", "vit_small_patch16_224"]
CHECK_PIPES  = ["K3", "K6", "K7"]
PLAN = [(m, p) for m in CHECK_MODELS for p in CHECK_PIPES]
TIME_BUDGET_HOURS = 10.0
SAVE_ALL_FOLD_WEIGHTS = []                        # only fold 0 is saved, as in Notebook 2

# ---- model and training settings: IDENTICAL to Notebook 2 ----
MODEL_NAME   = CHECK_MODELS[0]
PRETRAINED   = True
EPOCHS       = 25
BATCH_SIZE   = 32
LR           = 2e-4
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 1
LABEL_SMOOTHING = 0.1
FOLDS        = [0, 1, 2, 3, 4]
SEED         = 42
NUM_WORKERS  = 4

ON_KAGGLE = Path("/kaggle/input").is_dir()
INPUT_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path(r"H:\BD Weapon Inputs")
OUT_ROOT   = Path("/kaggle/working") if ON_KAGGLE else Path(r"H:\BD Weapon Training")
RESULTS    = OUT_ROOT / "ablation_check" / SCENARIO
RESULTS.mkdir(parents=True, exist_ok=True)
print("results ->", RESULTS)

results -> /kaggle/working/ablation_check/all


## 3. Pipeline definitions (as in Notebook 2)

In [3]:
REFERENCE = dict(bg="kept", size="letterbox", res=224, aug="light", color="rgb")
CHANGES = {
    "1": {},
    "2": dict(size="stretch"),
    "3": dict(res=384),
    "4": dict(aug="none"),
    "5": dict(aug="strong"),
    "6": dict(color="gray"),
    "7": dict(color="hsv"),
    "8": dict(color="lab"),
    "9": dict(color="clahe"),
}
PIPELINE_DEFS = {}
for k, ch in CHANGES.items():
    PIPELINE_DEFS["K" + k] = {**REFERENCE, **ch, "bg": "kept", "balanced": False, "tta": False}
    PIPELINE_DEFS["R" + k] = {**REFERENCE, **ch, "bg": "removed", "balanced": False, "tta": False}
PIPELINE_DEFS["F"] = {**REFERENCE, "balanced": True, "tta": True}   # placeholder, replaced in section 7
display(pd.DataFrame(PIPELINE_DEFS).T.drop(index="F"))

,bg,size,res,aug,color,balanced,tta
K1,kept,letterbox,224,light,rgb,False,False
R1,removed,letterbox,224,light,rgb,False,False
K2,kept,stretch,224,light,rgb,False,False
R2,removed,stretch,224,light,rgb,False,False
K3,kept,letterbox,384,light,rgb,False,False
R3,removed,letterbox,384,light,rgb,False,False
K4,kept,letterbox,224,none,rgb,False,False
R4,removed,letterbox,224,none,rgb,False,False
K5,kept,letterbox,224,strong,rgb,False,False
R5,removed,letterbox,224,strong,rgb,False,False


## 4. Inputs

In [4]:
def walk_find(base, predicate, skip=("standardized", "cutouts", "masks", "figures", "weights")):
    for dirpath, dirnames, filenames in os.walk(base, followlinks=True):
        d = Path(dirpath)
        hit = predicate(d, filenames)
        if hit:
            return hit
        dirnames[:] = [x for x in dirnames if x not in skip]
    return None

STAGE0 = walk_find(INPUT_ROOT, lambda d, f: d.parent if ("manifest.csv" in f and d.name == "metadata"
                                                        and d.parent.name == "stage0") else None)
assert STAGE0, "Stage 0 output not found. Add Input -> Your Work -> your Stage 0 notebook."
manifest = pd.read_csv(STAGE0 / "metadata" / "manifest.csv", dtype={"image_id": str})
CLASS_MAP = json.loads((STAGE0 / "metadata" / "class_map.json").read_text())
CLASSES = sorted(CLASS_MAP, key=CLASS_MAP.get)
NUM_CLASSES = len(CLASSES)
manifest["y"] = manifest.label.map(CLASS_MAP)
if "device" not in manifest:
    manifest["device"] = "unknown"
BGR = None                                         # only background-kept pipelines are run here

# Notebook 2 results of scenario "all" (read only: used for the K1 reference)
NB2_ALL = walk_find(INPUT_ROOT, lambda d, f: d if ("summary_all.csv" in f and d.name == "all"
                                                  and d.parent.name == "results") else None)
assert NB2_ALL, "Notebook 2 output (results/all) not found. Add Input -> Your Work -> Notebook 2, final version."
print("Stage 0:", STAGE0, "| Notebook 2 (all):", NB2_ALL)

# carry forward finished runs from an earlier version of THIS notebook
prev = walk_find(INPUT_ROOT, lambda d, f: d if ("summary_all.csv" in f and d.name == SCENARIO
                                               and d.parent.name == "ablation_check") else None)
if prev and prev.resolve() != RESULTS.resolve():
    n = 0
    for s in prev.rglob("*"):
        dst = RESULTS / s.relative_to(prev)
        if s.is_file() and not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(s, dst); n += 1
    print(f"Carried forward {n} files from {prev}")

test_df = manifest[manifest.split == "test"].reset_index(drop=True)
tv_df = manifest[manifest.split == "trainval"].reset_index(drop=True)
print(f"{len(tv_df)} train/val images, {len(test_df)} test images")

Stage 0: /kaggle/input/notebooks/arnicasarker/stage0-single-multi-3parts/stage0 | Notebook 2 (all): /kaggle/input/notebooks/arnicasarker/nb2-training-scenarios/results/all
2531 train/val images, 419 test images


## 5. Image preparation (as in Notebook 2)

In [5]:
PAD_GRAY = 128   # border colour for letterbox and for rotations (same gray as the cut-out background)

def load_rgb(row, bg):
    path = (BGR / row.cutout_path) if bg == "removed" else (STAGE0 / row.path)
    return np.asarray(Image.open(path).convert("RGB"))

def clahe_rgb(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def geometric(rgb, size, res):
    h, w = rgb.shape[:2]
    if size == "stretch":
        return cv2.resize(rgb, (res, res), interpolation=cv2.INTER_AREA)
    s = res / max(h, w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    small = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
    out = np.full((res, res, 3), PAD_GRAY, np.uint8)
    top, left = (res - nh) // 2, (res - nw) // 2
    out[top:top + nh, left:left + nw] = small
    return out

def build_cache(df, cfg):
    arrs = []
    for r in tqdm(df.itertuples(index=False), total=len(df), desc=f"preparing {cfg['bg']} {cfg['size']} {cfg['res']}", leave=False):
        rgb = load_rgb(r, cfg["bg"])
        if cfg["color"] == "clahe":
            rgb = clahe_rgb(rgb)
        arrs.append(geometric(rgb, cfg["size"], cfg["res"]))
    return np.stack(arrs)

def convert_color(rgb, color):
    # rgb: HxWx3 uint8 -> HxWx3 uint8 in the chosen representation
    if color == "gray":
        g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        return np.repeat(g[..., None], 3, axis=2)
    if color == "hsv":
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV_FULL)
    if color == "lab":
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    return rgb   # "rgb" and "clahe" (already applied in the cache)

IMAGENET_MEAN, IMAGENET_STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

def channel_stats(cache, color):
    # ImageNet statistics for RGB-like inputs; own statistics (from training images) for HSV and LAB
    if color in ("rgb", "clahe", "gray"):
        return IMAGENET_MEAN, IMAGENET_STD
    x = np.stack([convert_color(im, color) for im in cache[:: max(1, len(cache) // 300)]]).astype(np.float32) / 255
    return x.mean(axis=(0, 1, 2)), x.std(axis=(0, 1, 2)) + 1e-6

def train_augment(aug):
    fill = [PAD_GRAY] * 3
    if aug == "none":
        return T.Identity()
    if aug == "light":
        return T.Compose([T.RandomHorizontalFlip(), T.RandomRotation(15, fill=fill),
                          T.ColorJitter(brightness=0.2, contrast=0.2)])
    if aug == "strong":
        return T.Compose([T.RandomHorizontalFlip(), T.RandAugment(num_ops=2, magnitude=9, fill=fill)])
    raise ValueError(aug)

class ImageSet(Dataset):
    def __init__(self, cache, labels, color, mean, std, augment=None, erase=False, hflip=False):
        self.cache, self.labels, self.color = cache, labels, color
        self.mean = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(3, 1, 1)
        self.augment, self.hflip = augment, hflip
        self.erase = T.RandomErasing(p=0.25, value=0) if erase else None
    def __len__(self):
        return len(self.cache)
    def __getitem__(self, i):
        x = torch.from_numpy(self.cache[i]).permute(2, 0, 1)            # uint8 CxHxW
        if self.augment is not None:
            x = self.augment(x)
        if self.hflip:
            x = torch.flip(x, dims=[2])
        x = convert_color(x.permute(1, 2, 0).numpy().copy(), self.color)
        x = (torch.from_numpy(x).permute(2, 0, 1).float() / 255 - self.mean) / self.std
        if self.erase is not None:
            x = self.erase(x)
        return x, int(self.labels[i])

## 6. Training and evaluation (as in Notebook 2)

In [6]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def make_model(res):
    kw = dict(pretrained=PRETRAINED, num_classes=NUM_CLASSES)
    if MODEL_NAME.startswith(("vit", "deit", "swin")):
        kw["img_size"] = res
    return timm.create_model(MODEL_NAME, **kw)

def predict(model, cache, labels, color, mean, std, tta):
    model.eval()
    views = [False, True] if tta else [False]
    probs = 0
    for flip in views:
        dl = DataLoader(ImageSet(cache, labels, color, mean, std, hflip=flip), batch_size=BATCH_SIZE * 2,
                        shuffle=False, num_workers=NUM_WORKERS)
        out = []
        with torch.no_grad(), torch.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
            for x, _ in dl:
                out.append(F_.softmax(model(x.to(DEVICE)).float(), dim=1).cpu())
        probs = probs + torch.cat(out).numpy()
    return probs / len(views)

def train_fold(cfg, cache_tr, y_tr, cache_va, y_va, mean, std, seed):
    seed_everything(seed)
    ds = ImageSet(cache_tr, y_tr, cfg["color"], mean, std, augment=train_augment(cfg["aug"]),
                  erase=(cfg["aug"] == "strong"))
    if cfg["balanced"]:
        counts = np.bincount(y_tr, minlength=NUM_CLASSES)
        w = 1.0 / counts[y_tr]
        sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), len(y_tr), replacement=True)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, drop_last=True)
    else:
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    cutmix = T.CutMix(num_classes=NUM_CLASSES, alpha=1.0) if cfg["aug"] == "strong" else None

    model = make_model(cfg["res"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total = EPOCHS * len(dl); warm = WARMUP_EPOCHS * len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                              else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    log = []
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); tot, n = 0.0, 0
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if cutmix is not None and random.random() < 0.5:
                x, y = cutmix(x, y)
            with torch.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
                loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(x); n += len(x)
        log.append({"epoch": ep + 1, "train_loss": tot / max(n, 1), "sec": round(time.time() - t0, 1)})
    p_va = predict(model, cache_va, y_va, cfg["color"], mean, std, cfg["tta"])
    f1 = f1_score(y_va, p_va.argmax(1), average="macro")
    log[-1]["val_macro_f1"] = f1
    return model, p_va, pd.DataFrame(log)

def metrics(y, p):
    pred = p.argmax(1)
    return {"macro_f1": f1_score(y, pred, average="macro"), "accuracy": accuracy_score(y, pred),
            "balanced_acc": balanced_accuracy_score(y, pred)}

def save_confusion(y, pred, title, path):
    cm = confusion_matrix(y, pred, labels=range(NUM_CLASSES))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xticks(range(NUM_CLASSES), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(NUM_CLASSES), CLASSES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    fig.tight_layout(); fig.savefig(path, dpi=120); plt.close(fig)

def pred_frame(df, probs):
    return pd.DataFrame({"image_id": df.image_id.to_numpy(), "subset": df.subset.to_numpy(),
                         "device": df.device.to_numpy(), "y": df.y.to_numpy(), "pred": probs.argmax(1),
                         **{f"p_{c}": probs[:, i] for i, c in enumerate(CLASSES)}})

def run_pipeline(name, cfg=None, tv=None, te=None, extra=None):
    # name: folder name; cfg: pipeline settings (default PIPELINE_DEFS[name]);
    # tv / te: train-val and test tables (default: the scenario's); extra: {label: table} predicted by the test ensemble
    cfg = cfg or PIPELINE_DEFS[name]
    tv = tv_df if tv is None else tv.reset_index(drop=True)
    te = test_df if te is None else te.reset_index(drop=True)
    extra = extra or {}
    out = RESULTS / MODEL_NAME / name
    if (out / "summary.json").exists():
        print(f"{MODEL_NAME}/{name}: already done, skipping"); return
    out.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {SCENARIO} | {MODEL_NAME} / {name}: {cfg} ===")
    t_start = time.time()
    cache_tv, cache_te = build_cache(tv, cfg), build_cache(te, cfg)
    cache_ex = {k: build_cache(d, cfg) for k, d in extra.items()}
    y_tv, y_te = tv.y.to_numpy(), te.y.to_numpy()

    oof = np.zeros((len(tv), NUM_CLASSES)); test_probs = 0; fold_rows, logs = [], []
    ex_probs = {k: 0 for k in extra}
    norm = {}
    for k in FOLDS:
        tr, va = np.flatnonzero(tv.fold != k), np.flatnonzero(tv.fold == k)
        mean, std = channel_stats(cache_tv[tr], cfg["color"])
        norm[str(k)] = {"mean": [float(v) for v in mean], "std": [float(v) for v in std]}
        model, p_va, log = train_fold(cfg, cache_tv[tr], y_tv[tr], cache_tv[va], y_tv[va], mean, std, SEED + k)
        oof[va] = p_va
        test_probs = test_probs + predict(model, cache_te, y_te, cfg["color"], mean, std, cfg["tta"])
        for key, d in extra.items():
            ex_probs[key] = ex_probs[key] + predict(model, cache_ex[key], d.y.to_numpy(), cfg["color"], mean, std, cfg["tta"])
        m = metrics(y_tv[va], p_va); m.update(fold=k, n_val=len(va)); fold_rows.append(m)
        logs.append(log.assign(fold=k))
        print(f"  fold {k}: macro-F1 {m['macro_f1']:.3f} | acc {m['accuracy']:.3f} | "
              f"{log.sec.sum() / 60:.1f} min")
        if k == FOLDS[0] or name in SAVE_ALL_FOLD_WEIGHTS:   # weights for Grad-CAM and robustness in Notebook 3
            (out / "weights").mkdir(exist_ok=True)
            torch.save({k2: v.half() for k2, v in model.state_dict().items()}, out / "weights" / f"fold{k}.pt")
        del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    test_probs = test_probs / len(FOLDS)
    (out / "weights").mkdir(exist_ok=True)
    (out / "weights" / "norm.json").write_text(json.dumps(norm, indent=2))   # input normalisation per fold

    folds = pd.DataFrame(fold_rows); folds.to_csv(out / "fold_metrics.csv", index=False)
    pd.concat(logs).to_csv(out / "train_log.csv", index=False)
    in_folds = tv.fold.isin(FOLDS).to_numpy()
    pred_frame(tv, oof).assign(fold=tv.fold.to_numpy())[in_folds].to_csv(out / "oof_predictions.csv", index=False)
    pred_frame(te, test_probs).to_csv(out / "test_predictions.csv", index=False)
    extra_summary = {}
    for key, d in extra.items():
        p = ex_probs[key] / len(FOLDS)
        pred_frame(d, p).to_csv(out / f"{key}_predictions.csv", index=False)
        mk = metrics(d.y.to_numpy(), p)
        extra_summary.update({f"{key}_macro_f1": mk["macro_f1"], f"{key}_accuracy": mk["accuracy"], f"{key}_n": len(d)})
        save_confusion(d.y.to_numpy(), p.argmax(1), f"{MODEL_NAME} {name}: {key}", out / f"confusion_{key}.png")
    oof_m, test_m = metrics(y_tv[in_folds], oof[in_folds]), metrics(y_te, test_probs)
    per_class = f1_score(y_te, test_probs.argmax(1), average=None, labels=range(NUM_CLASSES))
    save_confusion(y_tv[in_folds], oof[in_folds].argmax(1), f"{MODEL_NAME} {name}: cross-validation", out / "confusion_cv.png")
    save_confusion(y_te, test_probs.argmax(1), f"{MODEL_NAME} {name}: test set", out / "confusion_test.png")

    summary = {"scenario": SCENARIO, "model": MODEL_NAME, "pipeline": name, **{f"cfg_{k}": v for k, v in cfg.items()},
               "cv_macro_f1_mean": folds.macro_f1.mean(), "cv_macro_f1_std": folds.macro_f1.std(ddof=1) if len(folds) > 1 else 0.0,
               "cv_accuracy_mean": folds.accuracy.mean(), "oof_macro_f1": oof_m["macro_f1"],
               "test_macro_f1": test_m["macro_f1"], "test_accuracy": test_m["accuracy"],
               "test_balanced_acc": test_m["balanced_acc"],
               **{f"test_f1_{c}": float(v) for c, v in zip(CLASSES, per_class)}, **extra_summary,
               "n_trainval": len(tv), "n_test": len(te),
               "epochs": EPOCHS, "folds": len(FOLDS), "minutes": round((time.time() - t_start) / 60, 1)}
    (out / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
    all_p = RESULTS / "summary_all.csv"
    row = pd.DataFrame([summary])
    (pd.concat([pd.read_csv(all_p), row]) if all_p.exists() else row).to_csv(all_p, index=False)
    print(f"  => CV macro-F1 {summary['cv_macro_f1_mean']:.3f} +/- {summary['cv_macro_f1_std']:.3f} | "
          f"test macro-F1 {summary['test_macro_f1']:.3f}"
          + "".join(f" | {k.replace('_macro_f1', '')} macro-F1 {v:.3f}" for k, v in extra_summary.items() if k.endswith("macro_f1"))
          + f" | {summary['minutes']} min")

## 7. Run

In [7]:
MODEL_COST = {"convnext_tiny": 1.3, "vit_small_patch16_224": 0.8}
def cost(model, cfg):
    hi = cfg["res"] >= 384
    return MODEL_COST.get(model, 1.0) * ((3.5 if model.startswith("vit") else 2.5) if hi else 1.0)

T_SESSION = time.time(); unit_min = None
for MODEL_NAME, name in PLAN:
    if (RESULTS / MODEL_NAME / name / "summary.json").exists():
        continue
    c = cost(MODEL_NAME, PIPELINE_DEFS[name]) * len(tv_df) / 2500
    elapsed_h = (time.time() - T_SESSION) / 3600
    if elapsed_h + (unit_min or 0) * c / 60 > TIME_BUDGET_HOURS:
        print(f"Time budget reached ({elapsed_h:.1f} h used). Next: {MODEL_NAME}/{name}."); break
    t0 = time.time()
    run_pipeline(name)
    unit_min = (time.time() - t0) / 60 / c

left = [f"{m}/{p}" for m, p in PLAN if not (RESULTS / m / p / "summary.json").exists()]
print(f"\nSession time: {(time.time() - T_SESSION) / 3600:.2f} h")
print(f"NOT FINISHED: {left} -> add this notebook's latest version as input and commit again." if left else "ALL DONE")


=== all | convnext_tiny / K3: {'bg': 'kept', 'size': 'letterbox', 'res': 384, 'aug': 'light', 'color': 'rgb', 'balanced': False, 'tta': False} ===


preparing kept letterbox 384:   0%|          | 0/2531 [00:00<?, ?it/s]

preparing kept letterbox 384:   0%|          | 0/419 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_23/250340227.py:54: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


... (64 outputs left out)

/tmp/ipykernel_23/250340227.py:54: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 3: macro-F1 0.992 | acc 0.992 | 3.2 min


/tmp/ipykernel_23/250340227.py:54: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 4: macro-F1 0.980 | acc 0.980 | 3.2 min
  => CV macro-F1 0.991 +/- 0.009 | test macro-F1 0.990 | 20.1 min

Session time: 3.74 h
ALL DONE


## Result table

Each pipeline is compared with K1 of the same model over the same five folds (paired t-test, Holm correction over the six new comparisons). ResNet-50 rows are read from Notebook 2 for reference only.

In [8]:
from scipy import stats
NAMES = {"K1": "Reference (letterbox 224, RGB)", "K3": "Letterbox 384", "K6": "Grayscale", "K7": "HSV"}
SHORT = {"resnet50": "ResNet-50", "convnext_tiny": "ConvNeXt-Tiny", "vit_small_patch16_224": "ViT-S/16"}

def folds_of(model, pipe):
    for base in (RESULTS, NB2_ALL):
        p = base / model / pipe / "fold_metrics.csv"
        if p.exists():
            return pd.read_csv(p).set_index("fold").macro_f1.reindex(FOLDS)
    return None

rows = []
for model in ["resnet50"] + CHECK_MODELS:
    ref = folds_of(model, "K1")
    if ref is None:
        print("K1 not found for", model); continue
    rows.append({"model": SHORT[model], "pipeline": "K1", "setting": NAMES["K1"], "cv_macro_f1": ref.mean(),
                 "cv_std": ref.std(ddof=1), "diff_vs_K1": np.nan, "better_folds": "", "p": np.nan})
    for pipe in CHECK_PIPES:
        s = folds_of(model, pipe)
        if s is None:
            continue
        d = s - ref
        p = stats.ttest_rel(s, ref).pvalue if d.abs().sum() > 0 else 1.0
        rows.append({"model": SHORT[model], "pipeline": pipe, "setting": NAMES[pipe], "cv_macro_f1": s.mean(),
                     "cv_std": s.std(ddof=1), "diff_vs_K1": d.mean(), "better_folds": f"{int((d > 0).sum())}/5", "p": p})
tab = pd.DataFrame(rows)

# Holm correction over all non-reference comparisons
m = tab.p.notna() & tab.model.isin([SHORT[x] for x in CHECK_MODELS])   # ResNet-50 rows are for reference only
order = tab[m].p.sort_values().index
k = len(order); holm, run_max = {}, 0.0
for i, idx in enumerate(order):
    run_max = max(run_max, min(1.0, (k - i) * tab.at[idx, "p"])); holm[idx] = run_max
tab["p_holm"] = pd.Series(holm)
tab["significant"] = tab.p_holm < 0.05

# test macro-F1 for the new runs and K1 (from summary files)
def test_f1(model, pipe):
    for base in (RESULTS, NB2_ALL):
        p = base / model / pipe / "summary.json"
        if p.exists():
            return json.loads(p.read_text())["test_macro_f1"]
    return np.nan
inv = {v: k for k, v in SHORT.items()}
tab["test_macro_f1"] = [test_f1(inv[r.model], r.pipeline) for r in tab.itertuples()]

tab.round(4).to_csv(RESULTS.parent / "ablation_check_table.csv", index=False)
display(tab.round(3))

for mdl, g in tab.groupby("model", sort=False):
    g = g.set_index("pipeline")
    if {"K3", "K6", "K7"} <= set(g.index):
        print(f"{mdl}: K3 {g.at['K3','diff_vs_K1']:+.3f} | gray {g.at['K6','diff_vs_K1']:+.3f} | "
              f"HSV {g.at['K7','diff_vs_K1']:+.3f} vs K1")

,model,pipeline,setting,cv_macro_f1,cv_std,diff_vs_K1,better_folds,p,p_holm,significant,test_macro_f1
0,ResNet-50,K1,"Reference (letterbox 224, RGB)",0.995,0.003,NaN,,NaN,NaN,False,0.995
1,ResNet-50,K3,Letterbox 384,0.998,0.002,0.003,4/5,0.103,NaN,False,0.998
2,ResNet-50,K6,Grayscale,0.985,0.004,-0.009,0/5,0.014,NaN,False,0.995
3,ResNet-50,K7,HSV,0.987,0.002,-0.008,0/5,0.005,NaN,False,0.992
4,ConvNeXt-Tiny,K1,"Reference (letterbox 224, RGB)",0.994,0.003,NaN,,NaN,NaN,False,0.998
5,ConvNeXt-Tiny,K3,Letterbox 384,0.998,0.002,0.004,4/5,0.022,0.094,False,0.998
6,ConvNeXt-Tiny,K6,Grayscale,0.988,0.002,-0.007,0/5,0.007,0.041,True,0.995
7,ConvNeXt-Tiny,K7,HSV,0.992,0.003,-0.002,2/5,0.219,0.658,False,0.993
8,ViT-S/16,K1,"Reference (letterbox 224, RGB)",0.995,0.002,NaN,,NaN,NaN,False,0.995
9,ViT-S/16,K3,Letterbox 384,0.996,0.005,0.001,3/5,0.706,0.706,False,0.998


ResNet-50: K3 +0.003 | gray -0.009 | HSV -0.008 vs K1
ConvNeXt-Tiny: K3 +0.004 | gray -0.007 | HSV -0.002 vs K1
ViT-S/16: K3 +0.001 | gray -0.012 | HSV -0.004 vs K1


In [9]:
import zipfile
z_path = OUT_ROOT / "ablation_check.zip"
with zipfile.ZipFile(z_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in (RESULTS.parent).rglob("*"):
        if f.is_file() and "weights" not in f.parts:
            z.write(f, f.relative_to(RESULTS.parent))
print("Download:", z_path, "(weights are kept in the output but not zipped)")

Download: /kaggle/working/ablation_check.zip (weights are kept in the output but not zipped)
